In [1]:
import sys
import os

model_upgrading_path = os.path.join("..","src")
sys.path.append(model_upgrading_path)

In [2]:
# moduel
from my_package.data.select_ship_dataset import get_ship_dataframe_from_database 
from my_package.data.load_database import load_database 

# basic
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
def get_sensor_pred_table(ship_id, col):
    
    # 테이블과 모델명을 사전으로 정의하여 코드 간결화
    table_names = {
        'CSU': 'tc_ai_csu_model_system_health_group',
        'STS': 'tc_ai_sts_model_system_health_group',
        'FTS': 'tc_ai_fts_model_system_health_group',
        'FMU': 'tc_ai_fmu_model_system_health_group',
        'CURRENT': 'tc_ai_electrode_model_group',
        'TRO': 'tc_ai_fault_model_group'
    }

    # 데이터베이스 이름
    database = 'ecs_dat1'
    
    # ship_id에 해당하는 각 테이블의 데이터프레임을 생성하여 사전에 저장
    dataframes = {name: get_ship_dataframe_from_database(table, database, ship_id)
                  for name, table in table_names.items()}
    
    # 데이터프레임 반환
    return dataframes[col]

In [4]:
def caculate_categorical_values(ship_id, col):
    
    # 센서 예측 데이터 프레임 임포트
    data = get_sensor_pred_table(ship_id, col)
    
    if col!='TRO':
        # 성능 지표 임계값
        indicator_dict = {'CSU':4, 'STS':4, 'FTS':8, 'FMU':5, 'CURRENT':8}

        # 센서 성능 지표 임계 값 선택 
        diff_indicator = indicator_dict[col]

        # 실제 값 - 예측 값 생성
        data['DIFF'] = abs(data['ACTUAL'] - data['PRED'])

        # 예측 값 카테고리 변수 생성
        data['CATEGORICAL_VALUE'] = data['DIFF'] <= diff_indicator
    else:
        abnormal_sum = data['STEEP_LABEL'] + data['OUT_OF_WATER_STEEP'] + data['TIME_OFFSET'] + data['SLOWLY_LABEL'] + data['HUNTING']
        data['abnormal_sum'] = abnormal_sum

        data['ACTUAL'] = 0
        data.loc[data['abnormal_sum']>=1,'ACTUAL'] =1

        data['CATEGORICAL_VALUE'] = data['ACTUAL']==data['PRED']

    return data

In [10]:
def caculate_performance_indicator(ship_id, cols):
    
    dataframs = {'SHIP_ID' : [] , 'SENCOR' : [], 'CORRECT' : [], 'TOTAL' : [], 'ACCURACY' : []}

    for col in cols:
        # 센서 예측 카테고리 데이터 프레임 생성 
        data = caculate_categorical_values(ship_id, col)
    
        correct = data['CATEGORICAL_VALUE'].sum()
        total = data['CATEGORICAL_VALUE'].count()
        accuracy = np.round(correct /  total,2) * 100

        dataframs['SHIP_ID'].append(ship_id)
        dataframs['SENCOR'].append(col)
        dataframs['CORRECT'].append(correct)
        dataframs['TOTAL'].append(total)
        dataframs['ACCURACY'].append(accuracy)
        
    accuracy_df = pd.DataFrame(dataframs)
    
    # 현재 날짜 가져오기
    current_date_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # 문자열을 다시 datetime 객체로 변환
    current_date = datetime.strptime(current_date_str, "%Y-%m-%d %H:%M:%S")
    
    accuracy_df['DT_UPDATE'] = current_date
    return accuracy_df

In [18]:
def load_performance_dataset(ship_id):

    cols = ['CSU','STS','FTS','FMU','CURRENT','TRO']
    accuracy_df = caculate_performance_indicator(ship_id, cols) # 업데이트 날짜 추가
    
    load_database(accuracy_df,'tc_ai_model_accuracy_table')

In [20]:
load_performance_dataset('T20191002002')

DataFrame has been successfully loaded into tc_ai_model_accuracy_table table in ecs_dat1 database.
